# BOJ Scenario Analysis (V3 - Forward Fill Missing Data)

改善点:
1. **欠損値の補完**: 元データに欠損がある場合、前日の値を引き継ぎます（ffill）。
2. **特徴量設計**: 全BOJスプレッドの前日値と5日MA乖離、その他項目の5日MA乖離を使用。
3. **シナリオ分析**: 次回会合の利上げ有無（Hikeフラグ）を特徴量に含めます。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. データの読み込みと欠損値補完
df = pd.read_excel('data/BOJ_data.xlsx')
df = df.iloc[1:].copy()
df['日付'] = pd.to_datetime(df['日付'], format='%Y年%m月%d日')

swap_cols = [f'JPBOJ{i}ONI=TRDT (MID_PRICE)' for i in range(1, 9)]
tona_col = 'JPY1DOIS=ICAP (MID_PRICE)'
jpy_col = 'JPY= (MID_PRICE)'
market_cols = ['JGBc1 (TRDPRC_1)', '.N225 (TRDPRC_1)', '.DXY (TRDPRC_1)']
all_val_cols = swap_cols + [tona_col, jpy_col] + market_cols

for col in all_val_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values('日付').reset_index(drop=True)

# 【新規】元データの欠損値を前日の値で埋める
df[all_val_cols] = df[all_val_cols].ffill()

# データの先頭部分でffillできない(前日がない)欠損は削除
df.dropna(subset=[swap_cols[0], tona_col], inplace=True)
print(f'Loaded and Forward-filled: {len(df)} rows')

In [ ]:
# 2. MPM結果の定義 (実績に合わせて修正してください)
mpm_results = {
    '2024-01-23': 0, '2024-03-19': 1, '2024-04-26': 0, '2024-06-14': 0, 
    '2024-07-31': 1, '2024-09-20': 0, '2024-10-31': 0, '2024-12-19': 0, 
    '2025-01-24': 1, '2025-03-19': 0, '2025-04-30': 0, '2025-06-17': 0, 
    '2025-07-31': 0, '2025-09-19': 0, '2025-10-30': 0, '2025-12-19': 0,
    '2026-01-23': 0, '2026-03-19': 0, '2026-04-28': 0, '2026-06-16': 0
}
mpm_dates = pd.to_datetime(list(mpm_results.keys()))
def get_next_mpm_info(d):
    future = mpm_dates[mpm_dates > d]
    if len(future) > 0:
        next_date = future[0]
        return next_date, mpm_results[next_date.strftime('%Y-%m-%d')]
    return None, None
res = df['日付'].apply(get_next_mpm_info)
df['Next_MPM'] = [x[0] for x in res]
df['Next_MPM_Hike'] = [x[1] for x in res]
df['DaysToNextMPM'] = (df['Next_MPM'] - df['日付']).dt.days

In [ ]:
# 3. 特徴量生成
features = []
df_feats = df.copy()
window = 5

for i, col in enumerate(swap_cols):
    spread_col = f'BOJ{i+1}_Spread'
    df_feats[spread_col] = df_feats[col] - df_feats[tona_col]
    df_feats[f'{spread_col}_lag1'] = df_feats[spread_col].shift(1)
    features.append(f'{spread_col}_lag1')
    df_feats[f'{spread_col}_diff_MA5'] = df_feats[f'{spread_col}_lag1'] - df_feats[spread_col].shift(1).rolling(5).mean()
    features.append(f'{spread_col}_diff_MA5')

for col in [jpy_col] + market_cols:
    df_feats[f'{col}_diff_MA5'] = df_feats[col].shift(1) - df_feats[col].shift(1).rolling(5).mean()
    features.append(f'{col}_diff_MA5')

features.append('DaysToNextMPM')
features.append('Next_MPM_Hike')

# ターゲット: 5日後のBOJ_3の変化幅
df_feats['Target'] = df_feats[swap_cols[2]].shift(-5) - df_feats[swap_cols[2]]

# MPM後除外
exclude = []
for mpm in mpm_dates: [exclude.append(mpm + pd.Timedelta(days=o)) for o in range(1, 6)]
df_feats = df_feats[~df_feats['日付'].isin(exclude)]

df_ready = df_feats.dropna(subset=['Target']).copy()
print(f'Total samples: {len(df_ready)}')
display(df_ready[features + ['Target']].head())

In [ ]:
# 4. 学習と評価
X = df_ready[features]
y = df_ready['Target']
tscv = TimeSeriesSplit(n_splits=5)
all_mae, all_dir = [], []

for tr, te in tscv.split(X):
    X_train, X_test = X.iloc[tr], X.iloc[te]
    y_train, y_test = y.iloc[tr], y.iloc[te]
    model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, random_state=42, verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='rmse', callbacks=[lgb.early_stopping(stopping_rounds=50)])
    y_pred = model.predict(X_test)
    all_mae.append(mean_absolute_error(y_test, y_pred))
    all_dir.append(np.mean(np.sign(y_test) == np.sign(y_pred)))

print(f'\n--- Scenario Analysis V3 (ffill) Results ---')
print(f'MAE: {np.mean(all_mae)*100:.3f} bps')
print(f'Direction Acc: {np.mean(all_dir):.2%}')

In [ ]:
# 5. 可視化
imp = pd.DataFrame({'f': features, 'i': model.booster_.feature_importance(importance_type='gain')}).sort_values('i', ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x='i', y='f', data=imp)
plt.title('Feature Importance (Gain)')
plt.show()